# Chapter 4, Exercise 2: WER before and after Arabic text normalization

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 4, Exercise 2.** A recognizer outputs الى المدرسه while the reference is إلى المدرسة. Compute the WER by hand before normalization and after applying the Table 4.2 rules, and state which rules you used. Then, in Python, write a short script that applies your normalization rules to both strings and recomputes the WER to confirm your manual result. Explain why these are scoring conventions rather than claims that the forms are identical.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_04_Exercise_02.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [1]:
!pip install -q jiwer

## 1. By hand

| Position | Reference | Hypothesis | Literal comparison |
|---|---|---|---|
| 1 | إلى | الى | differs (hamza below alif missing) |
| 2 | المدرسة | المدرسه | differs (ه instead of ة) |

Before normalization: S = 2, D = 0, I = 0, N = 2, so **WER = 2/2 = 100 %**.

After applying two optional Table 4.2 rules, *normalize alif forms* (إ to ا) and *normalize tāʾ marbūṭa and hāʾ* (ة to ه, or both to one chosen form), both strings become الى المدرسه: S = 0, **WER = 0 %**. The recognizer's output has not changed; only the scoring convention has.

## 2. In Python: a normalization script applied identically to both sides

Every rule is a named, switchable step so that the report can list exactly which ones were used.

In [2]:
import re, unicodedata, jiwer

DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalize(text, rules):
    """Apply the selected Table 4.2 rules. `rules` is a set of rule names."""
    t = unicodedata.normalize("NFC", text)
    if "remove_diacritics" in rules:   t = DIACRITICS.sub("", t)
    if "remove_tatweel" in rules:      t = t.replace("\u0640", "")
    if "normalize_alif" in rules:      t = re.sub("[أإآٱ]", "ا", t)
    if "normalize_ta_marbuta" in rules: t = t.replace("ة", "ه")     # chosen form: ه
    if "normalize_alif_maqsura" in rules: t = t.replace("ى", "ي")
    if "normalize_digits" in rules:    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    if "normalize_punct_space" in rules:
        t = re.sub(r"[\u060C\u061B\u061F\u066A-\u066D\u06D4!-/:-@\[-`{-~]", " ", t)
        t = re.sub(r"\s+", " ", t).strip()
    return t

reference  = "إلى المدرسة"
hypothesis = "الى المدرسه"

def report(rules):
    r, h = normalize(reference, rules), normalize(hypothesis, rules)
    o = jiwer.process_words(r, h)
    print(f"rules={sorted(rules) or 'none'}")
    print(f"  ref='{r}'  hyp='{h}'  S={o.substitutions} D={o.deletions} I={o.insertions} "
          f"N={len(r.split())}  WER={100*o.wer:.0f}%")

report(set())                                                   # before normalization
report({"remove_diacritics", "remove_tatweel", "normalize_punct_space"})   # "safe" rules only
report({"normalize_alif"})                                      # one merging rule
report({"normalize_alif", "normalize_ta_marbuta"})              # both merging rules -> confirms 0 %

rules=none
  ref='إلى المدرسة'  hyp='الى المدرسه'  S=2 D=0 I=0 N=2  WER=100%
rules=['normalize_punct_space', 'remove_diacritics', 'remove_tatweel']
  ref='إلى المدرسة'  hyp='الى المدرسه'  S=2 D=0 I=0 N=2  WER=100%
rules=['normalize_alif']
  ref='الى المدرسة'  hyp='الى المدرسه'  S=1 D=0 I=0 N=2  WER=50%
rules=['normalize_alif', 'normalize_ta_marbuta']
  ref='الى المدرسه'  hyp='الى المدرسه'  S=0 D=0 I=0 N=2  WER=0%


Output of the last line confirms the hand result: with *normalize alif forms* and *normalize tāʾ marbūṭa and hāʾ*, WER = 0 %. With only the "usually safe" rules (diacritics, tatweel, punctuation and spacing) the WER stays at 100 %, because neither difference in this pair involves those rules; with alif normalization alone it is 50 %.

## 3. Why these are scoring conventions, not claims of identity

* إلى and الى are the same word spelled with and without the hamza seat; the rule reflects a real informal-writing habit, so merging them is defensible for undiacritized transcription. But the same rule also merges genuinely different words that differ only in hamza (Table 4.2 lists أ, إ, آ, ٱ), so it destroys written distinctions.
* ة and ه are **not** generally equivalent: المدرسة 'the school' with ة and مدرسه 'his teacher' (مُدَرِّسُهُ) with ه are different words with different pronunciations (Section 4.2). Merging them is justified only when the transcription convention permits ه as an informal spelling of ة, as it does in much dialectal and social-media writing.
* A normalization rule therefore says "for the purpose of comparing systems on this benchmark, we agree not to count this difference." It is a decision about the *metric*, made to remove spelling noise, and the same script must be applied to both reference and hypothesis. It does not say the forms are the same word, which is why the rules, and ideally the script, must be published with every WER.